# M2-YOLO Surface Defect Fresh Train (Colab T4)

**목표**: yolov8m.pt → mAP50 **0.9+**

**Drive 준비물 (실행 전 업로드 필수):**
1. `surface.zip` — M2 데이터셋 (~700MB)

(M2는 fresh train이라 baseline best.pt 불필요. yolov8m.pt는 코랩에서 자동 다운로드)

**업로드 위치는 자유.** 셀 4의 `DRIVE_DIR` 변수만 본인 폴더로 변경.
- 기본값: `/content/drive/MyDrive/drone_inspect/`

**예상 시간:** 6~8시간 (T4 16GB, batch=16, 25ep)

**런타임:** 런타임 → 런타임 유형 변경 → GPU → T4

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. 패키지 설치

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 데이터셋 압축 해제

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ Drive에 surface.zip을 올린 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')
ZIP_NAME = 'surface.zip'

WORK = Path('/content/m2_train')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음 — Drive에 올리고 DRIVE_DIR 경로 확인'

ds_dir = WORK / 'surface'
needs_extract = (not ds_dir.exists()) or (not (ds_dir / 'images' / 'val').exists())
if needs_extract:
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting surface.zip ...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
else:
    print(f'Already extracted: {ds_dir}')

print('\n=== 데이터셋 카운트 ===')
for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        cnt = len([f for f in p.glob('*') if f.is_file()])
        print(f'  {split}: {cnt} images')
    else:
        print(f'  {split}: NOT FOUND ({p})')

## 5. data.yaml 생성

In [ ]:
yaml_text = f"""# M2 surface - Colab
path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: surface_defect_wall
  1: baseboard_defect
"""
data_yaml = WORK / 'surface.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## 6. Fresh Train (T4 batch=16, imgsz=960, 25ep)

In [ ]:
from ultralytics import YOLO

# Fresh train: yolov8m.pt에서 시작 (자동 다운로드)
model = YOLO('yolov8m.pt')

results = model.train(
    data=str(data_yaml),
    epochs=25,
    batch=16,             # T4 16GB → batch=16 (M5v2가 26%만 썼으니 배 가능)
    imgsz=960,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-3,             # fresh train: lr 큼 (fine-tune 1e-4보다 10배)
    lrf=0.01,
    patience=10,
    warmup_epochs=3,
    close_mosaic=15,
    # freeze 없음 (fresh train)
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    shear=2.0, perspective=0.001,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.05,
    erasing=0.0,
    copy_paste=0.3,
    multi_scale=0.2,
    save_period=5,
    plots=True,
    project='/content/runs',
    name='m2_train',
    exist_ok=True,
)

print('Train done.')

## 7. ONNX Export

In [ ]:
best_path = Path('/content/runs/m2_train/weights/best.pt')
print(f'best.pt: {best_path} ({best_path.stat().st_size/1024/1024:.1f}MB)')

best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)

onnx_path = best_path.with_suffix('.onnx')
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f}MB)')

## 8. 평가 (val set mAP)

In [ ]:
metrics = best_model.val(data=str(data_yaml), imgsz=960, batch=16)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall: {metrics.box.mr:.4f}')

## 9. 결과를 Google Drive에 저장

In [ ]:
OUT = DRIVE_DIR / 'm2_results'
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(onnx_path, OUT / 'm2_yolo_surface.onnx')
shutil.copy2(best_path, OUT / 'm2_best.pt')

results_csv = best_path.parent.parent / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, OUT / 'results.csv')

for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)

print('Saved to:', OUT)
for f in OUT.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)')

## 10. 완료 — 다음 단계 (로컬 PC)

1. Google Drive `m2_results/m2_yolo_surface.onnx` 다운로드
2. 로컬 `backend/models_weights/m2_yolo_surface.onnx`에 덮어쓰기
3. mAP50 확인 (results.csv 마지막 줄)